## Example 1: brotli functionality test 1

In [ ]:
# Example 1: Basic Brotli compression and decompression
import brotli
import os

print("=== Brotli Basic Compression and Decompression ===")

try:
    # Basic compression functionality
    print("1. Basic Compression:")
    
    # Test data for compression
    original_data = b"Hello, World! This is a test string for Brotli compression."
    print(f"Original data: {original_data}")
    print(f"Original size: {len(original_data)} bytes")
    
    # Compress the data
    compressed_data = brotli.compress(original_data)
    print(f"Compressed size: {len(compressed_data)} bytes")
    print(f"Compression ratio: {len(original_data)/len(compressed_data):.2f}:1")
    print(f"Space saved: {(1 - len(compressed_data)/len(original_data))*100:.1f}%")
    
    # Basic decompression
    print("\n2. Basic Decompression:")
    
    # Decompress the data
    decompressed_data = brotli.decompress(compressed_data)
    print(f"Decompressed data: {decompressed_data}")
    print(f"Decompressed size: {len(decompressed_data)} bytes")
    
    # Verify integrity
    is_identical = original_data == decompressed_data
    print(f"Data integrity check: {'✓ PASSED' if is_identical else '✗ FAILED'}")
    
    # Test with different data types
    print("\n3. Different Data Types:")
    
    test_cases = [
        (b"Simple text", "Simple text"),
        (b"A" * 100, "Repeated characters (100 A's)"),
        (b"The quick brown fox jumps over the lazy dog." * 5, "Repeated sentence"),
        (bytes(range(256)), "Binary data (0-255)")
    ]
    
    for data, description in test_cases:
        compressed = brotli.compress(data)
        decompressed = brotli.decompress(compressed)
        
        ratio = len(data) / len(compressed) if len(compressed) > 0 else 1
        success = data == decompressed
        
        print(f"  {description}:")
        print(f"    Original: {len(data)} bytes -> Compressed: {len(compressed)} bytes")
        print(f"    Ratio: {ratio:.2f}:1 | Integrity: {'✓' if success else '✗'}")
    
    # Module attributes and version
    print("\n4. Module Information:")
    attrs = [attr for attr in dir(brotli) if not attr.startswith('_')]
    print(f"Public attributes: {attrs}")
    
    # Check for version information
    version_info = getattr(brotli, '__version__', 'Unknown')
    print(f"Brotli version: {version_info}")
    
    print("\n✓ Basic compression and decompression completed successfully!")
    
except Exception as e:
    print(f"Error in basic functionality: {e}")
    raise AssertionError(
        f"brotli example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: brotli functionality test 2

In [ ]:
# Example 2: Compression levels and quality settings
import brotli
import time
import json

print("=== Brotli Compression Levels and Quality Settings ===")

try:
    # Test data - larger text for better compression demonstration
    test_text = """
    Brotli is a generic-purpose lossless compression algorithm that compresses data 
    using a combination of a modern variant of the LZ77 algorithm, Huffman coding 
    and 2nd order context modeling, with a compression ratio comparable to the best 
    currently available general-purpose compression methods. It is similar in speed 
    with deflate but offers more dense compression. The specification of the Brotli 
    Compressed Data Format is defined in RFC 7932. Brotli is used in WOFF 2.0 web 
    fonts, and is supported by all major web browsers.
    """ * 10  # Repeat for better compression demonstration
    
    original_data = test_text.encode('utf-8')
    print(f"Original data size: {len(original_data):,} bytes")
    
    # Test different compression qualities (0-11)
    print("\n1. Compression Quality Levels (0-11):")
    
    quality_results = []
    for quality in range(0, 12, 2):  # Test every other quality level
        start_time = time.time()
        
        # Compress with specific quality
        compressed = brotli.compress(original_data, quality=quality)
        
        compression_time = time.time() - start_time
        
        # Test decompression
        start_time = time.time()
        decompressed = brotli.decompress(compressed)
        decompression_time = time.time() - start_time
        
        # Calculate metrics
        ratio = len(original_data) / len(compressed)
        integrity = original_data == decompressed
        
        result = {
            'quality': quality,
            'compressed_size': len(compressed),
            'ratio': ratio,
            'compression_time': compression_time * 1000,  # Convert to ms
            'decompression_time': decompression_time * 1000,  # Convert to ms
            'integrity': integrity
        }
        
        quality_results.append(result)
        
        print(f"  Quality {quality:2d}: {len(compressed):,} bytes | "
              f"Ratio: {ratio:.2f}:1 | "
              f"Compress: {compression_time*1000:.2f}ms | "
              f"Decompress: {decompression_time*1000:.2f}ms | "
              f"{'✓' if integrity else '✗'}")
    
    # Analyze quality vs performance trade-offs
    print("\n2. Quality vs Performance Analysis:")
    
    best_ratio = max(quality_results, key=lambda x: x['ratio'])
    fastest_compress = min(quality_results, key=lambda x: x['compression_time'])
    fastest_decompress = min(quality_results, key=lambda x: x['decompression_time'])
    
    print(f"  Best compression ratio: Quality {best_ratio['quality']} ({best_ratio['ratio']:.2f}:1)")
    print(f"  Fastest compression: Quality {fastest_compress['quality']} ({fastest_compress['compression_time']:.2f}ms)")
    print(f"  Fastest decompression: Quality {fastest_decompress['quality']} ({fastest_decompress['decompression_time']:.2f}ms)")
    
    # Window size parameter (lgwin) if available
    print("\n3. Window Size Parameter (lgwin):")
    
    # Test different window sizes (16-24, default is 22)
    window_sizes = [16, 20, 22, 24]
    
    for lgwin in window_sizes:
        try:
            compressed = brotli.compress(original_data, quality=6, lgwin=lgwin)
            decompressed = brotli.decompress(compressed)
            
            ratio = len(original_data) / len(compressed)
            integrity = original_data == decompressed
            
            print(f"  Window size 2^{lgwin} ({2**lgwin:,}): {len(compressed):,} bytes | "
                  f"Ratio: {ratio:.2f}:1 | {'✓' if integrity else '✗'}")
        except Exception as e:
            print(f"  Window size 2^{lgwin}: Error - {e}")
            raise
    
    # Different data types and compression efficiency
    print("\n4. Compression Efficiency by Data Type:")
    
    data_types = {
        "Plain Text": "Lorem ipsum dolor sit amet, consectetur adipiscing elit. " * 50,
        "JSON Data": json.dumps({f"key_{i}": f"value_{i}" for i in range(100)}, indent=2),
        "Repeated Pattern": "ABCDEFGH" * 100,
        "Random-like": "xKj9mP2qR5wT8nL3vB7cF1zA4yE6uI0oS9hG",
        "HTML Content": "<html><body><h1>Title</h1><p>Content paragraph.</p></body></html>" * 25
    }
    
    for data_type, content in data_types.items():
        data_bytes = content.encode('utf-8')
        
        # Test with medium quality (6)
        compressed = brotli.compress(data_bytes, quality=6)
        decompressed = brotli.decompress(compressed)
        
        ratio = len(data_bytes) / len(compressed)
        space_saved = (1 - len(compressed) / len(data_bytes)) * 100
        
        print(f"  {data_type:15}: {len(data_bytes):,} -> {len(compressed):,} bytes | "
              f"Ratio: {ratio:.2f}:1 | Saved: {space_saved:.1f}%")
    
    # Memory usage estimation with large data
    print("\n5. Large Data Performance:")
    
    large_data = b"Testing large data compression performance. " * 10000  # ~430KB
    
    start_time = time.time()
    compressed_large = brotli.compress(large_data, quality=4)
    compress_time = time.time() - start_time
    
    start_time = time.time()
    decompressed_large = brotli.decompress(compressed_large)
    decompress_time = time.time() - start_time
    
    ratio_large = len(large_data) / len(compressed_large)
    
    print(f"  Large data ({len(large_data):,} bytes): {len(compressed_large):,} compressed bytes")
    print(f"  Compression ratio: {ratio_large:.2f}:1")
    print(f"  Compression time: {compress_time:.3f}s ({len(large_data)/compress_time/1024/1024:.1f} MB/s)")
    print(f"  Decompression time: {decompress_time:.3f}s ({len(large_data)/decompress_time/1024/1024:.1f} MB/s)")
    print(f"  Data integrity: {'✓' if large_data == decompressed_large else '✗'}")
    
    print("\n✓ Compression levels and quality settings completed successfully!")
    
except Exception as e:
    print(f"Error in quality settings: {e}")
    raise AssertionError(
        f"brotli example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: brotli functionality test 3

In [ ]:
# Example 3: Streaming compression and advanced features
import brotli
import tempfile
import os

print("=== Brotli Streaming Compression and Advanced Features ===")

try:
    # Streaming compression using Compressor class
    print("1. Streaming Compression with Compressor:")
    
    # Check if Compressor class is available
    if hasattr(brotli, 'Compressor'):
        # Create a compressor object
        compressor = brotli.Compressor(quality=6)
        print("Compressor created with quality=6")
        
        # Simulate streaming data
        data_chunks = [
            b"First chunk of streaming data. ",
            b"Second chunk with more information. ",
            b"Third chunk containing different content. ",
            b"Fourth and final chunk to complete the stream."
        ]
        
        compressed_chunks = []
        total_input_size = 0
        
        print("Processing data chunks:")
        for i, chunk in enumerate(data_chunks):
            # Process each chunk
            compressed_chunk = compressor.process(chunk)
            compressed_chunks.append(compressed_chunk)
            total_input_size += len(chunk)
            
            print(f"  Chunk {i+1}: {len(chunk)} bytes -> {len(compressed_chunk)} compressed bytes")
        
        # Finish compression
        final_chunk = compressor.finish()
        compressed_chunks.append(final_chunk)
        print(f"  Final: {len(final_chunk)} bytes")
        
        # Combine all compressed data
        complete_compressed = b''.join(compressed_chunks)
        total_compressed_size = len(complete_compressed)
        
        print(f"\nStreaming compression results:")
        print(f"  Total input: {total_input_size} bytes")
        print(f"  Total compressed: {total_compressed_size} bytes")
        print(f"  Compression ratio: {total_input_size/total_compressed_size:.2f}:1")
        
        # Verify by decompressing
        original_combined = b''.join(data_chunks)
        decompressed_stream = brotli.decompress(complete_compressed)
        stream_integrity = original_combined == decompressed_stream
        print(f"  Stream integrity: {'✓ PASSED' if stream_integrity else '✗ FAILED'}")
        
    else:
        print("Compressor class not available - using regular compression")
        
        # Fallback: simulate streaming with regular compression
        data_chunks = [
            b"First chunk of streaming data. ",
            b"Second chunk with more information. ",
            b"Third chunk containing different content. ",
            b"Fourth and final chunk to complete the stream."
        ]
        
        combined_data = b''.join(data_chunks)
        compressed_data = brotli.compress(combined_data, quality=6)
        
        print(f"  Simulated streaming: {len(combined_data)} -> {len(compressed_data)} bytes")
        print(f"  Ratio: {len(combined_data)/len(compressed_data):.2f}:1")
    
    # Streaming decompression using Decompressor class
    print("\n2. Streaming Decompression:")
    
    if hasattr(brotli, 'Decompressor'):
        print("Testing Decompressor class")
        
        # Create test compressed data
        test_data = b"This is test data for streaming decompression. " * 10
        full_compressed = brotli.compress(test_data, quality=6)
        
        # Create decompressor object
        decompressor = brotli.Decompressor()
        print("Decompressor created")
        
        # Split compressed data into chunks
        chunk_size = max(1, len(full_compressed) // 3)
        compressed_chunks = [
            full_compressed[i:i+chunk_size] 
            for i in range(0, len(full_compressed), chunk_size)
        ]
        
        decompressed_chunks = []
        print("Processing compressed chunks:")
        
        for i, chunk in enumerate(compressed_chunks):
            try:
                decompressed_chunk = decompressor.process(chunk)
                decompressed_chunks.append(decompressed_chunk)
                print(f"  Chunk {i+1}: {len(chunk)} -> {len(decompressed_chunk)} bytes")
            except Exception as e:
                print(f"  Chunk {i+1}: Processing error - {e}")
                raise
        
        # Check if finished
        is_finished = getattr(decompressor, 'is_finished', lambda: True)()
        print(f"  Decompression finished: {'✓' if is_finished else '✗'}")
        
        # Verify result
        stream_result = b''.join(decompressed_chunks)
        stream_match = test_data == stream_result
        print(f"  Stream integrity: {'✓ PASSED' if stream_match else '✗ FAILED'}")
        
    else:
        print("Decompressor class not available")
    
    # Advanced compression parameters
    print("\n3. Advanced Compression Parameters:")
    
    test_data = b"Advanced parameter testing with Brotli compression. " * 50
    
    # Test different parameter combinations
    parameter_tests = [
        {"quality": 1, "description": "Fastest compression"},
        {"quality": 6, "description": "Balanced compression"},
        {"quality": 11, "description": "Maximum compression"}
    ]
    
    # Add lgwin parameter if supported
    try:
        brotli.compress(test_data, quality=6, lgwin=20)
        parameter_tests.append({"quality": 6, "lgwin": 20, "description": "Custom window size"})
    except TypeError:
        print("  lgwin parameter not supported")
    
    # Add mode parameter if supported
    if hasattr(brotli, 'MODE_TEXT'):
        parameter_tests.append({"quality": 6, "mode": brotli.MODE_TEXT, "description": "Text mode optimization"})
    
    for params in parameter_tests:
        description = params.pop("description")
        
        try:
            compressed = brotli.compress(test_data, **params)
            decompressed = brotli.decompress(compressed)
            
            ratio = len(test_data) / len(compressed)
            integrity = test_data == decompressed
            
            print(f"  {description}:")
            print(f"    Parameters: {params}")
            print(f"    Result: {len(test_data)} -> {len(compressed)} bytes | Ratio: {ratio:.2f}:1 | {'✓' if integrity else '✗'}")
            
        except Exception as e:
            print(f"  {description}: Error - {e}")
            raise
        
        # Restore description for next iteration
        params["description"] = description
    
    # File I/O simulation
    print("\n4. File I/O Simulation:")
    
    # Create temporary file for testing
    with tempfile.NamedTemporaryFile(delete=False, suffix='.txt') as temp_input:
        file_content = b"File compression test content with various data patterns. " * 100
        temp_input.write(file_content)
        temp_input.flush()
        
        print(f"Created test file: {len(file_content)} bytes")
        
        # Read and compress file content
        with open(temp_input.name, 'rb') as f:
            file_data = f.read()
        
        compressed_file_data = brotli.compress(file_data, quality=6)
        
        # Write compressed data to another temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix='.br') as temp_compressed:
            temp_compressed.write(compressed_file_data)
            temp_compressed.flush()
            
            print(f"Compressed file: {len(compressed_file_data)} bytes")
            print(f"Compression ratio: {len(file_data)/len(compressed_file_data):.2f}:1")
            
            # Read back and decompress
            with open(temp_compressed.name, 'rb') as f:
                read_compressed = f.read()
            
            decompressed_file_data = brotli.decompress(read_compressed)
            file_integrity = file_data == decompressed_file_data
            
            print(f"File round-trip integrity: {'✓ PASSED' if file_integrity else '✗ FAILED'}")
            
            # Clean up
            os.unlink(temp_input.name)
            os.unlink(temp_compressed.name)
    
    # Error handling and edge cases
    print("\n5. Error Handling and Edge Cases:")
    
    edge_cases = [
        (b"", "Empty data"),
        (b"x", "Single byte"),
        (b"abc" * 1000, "Repetitive pattern"),
        (bytes([i % 256 for i in range(1000)]), "Sequential pattern")
    ]
    
    for data, description in edge_cases:
        try:
            compressed = brotli.compress(data, quality=6)
            decompressed = brotli.decompress(compressed)
            
            if len(data) > 0:
                ratio = len(data) / len(compressed)
            else:
                ratio = float('inf') if len(compressed) == 0 else 0
            
            integrity = data == decompressed
            
            print(f"  {description:18}: {len(data):4} -> {len(compressed):4} bytes | "
                  f"{'∞' if ratio == float('inf') else f'{ratio:.2f}'}:1 | "
                  f"{'✓' if integrity else '✗'}")
            
        except Exception as e:
            print(f"  {description:18}: Error - {e}")
            raise
    
    # Invalid data handling
    print("\n6. Invalid Data Handling:")
    
    invalid_cases = [
        (b"not compressed", "Plain text"),
        (b"\x00\x01\x02", "Random bytes"),
        (b"", "Empty data"),
        (b"\xff" * 50, "High entropy")
    ]
    
    for invalid_data, description in invalid_cases:
        # The decoder must REJECT each input with brotli.error; accepting one is a failure.
        try:
            result = brotli.decompress(invalid_data)
        except brotli.error as e:
            assert "decoder failed" in str(e), f"{description}: unexpected brotli.error message: {e}"
            print(f"  {description:15}: {type(e).__name__} (expected)")
        else:
            raise AssertionError(f"{description}: brotli.decompress accepted invalid data ({len(result)} bytes)")
    
    print("\n✓ Streaming compression and advanced features completed successfully!")
    
except Exception as e:
    print(f"Error in streaming compression: {e}")
    raise AssertionError(
        f"brotli example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: brotli functionality test 4

In [ ]:
# Example 4: Performance comparison and real-world applications
import brotli
import time
import gzip
import zlib
import json
import hashlib

print("=== Brotli Performance and Real-World Applications ===")

try:
    # Compression algorithm comparison
    print("1. Compression Algorithm Comparison:")
    
    # Create test datasets with different characteristics
    test_datasets = {
        "Text": ("The quick brown fox jumps over the lazy dog. " * 100).encode('utf-8'),
        "JSON": json.dumps([{"id": i, "name": f"user_{i}", "data": list(range(10))} for i in range(50)], indent=2).encode('utf-8'),
        "Repetitive": ("ABCDEFGHIJ" * 200).encode('utf-8'),
        "Binary": bytes([(i * 13 + 7) % 256 for i in range(1000)])
    }
    
    print(f"{'Dataset':<12} | {'Original':<8} | {'Brotli':<10} | {'Gzip':<10} | {'Zlib':<10} | {'Best'}")
    print("-" * 70)
    
    for name, data in test_datasets.items():
        original_size = len(data)
        
        # Brotli compression
        brotli_compressed = brotli.compress(data, quality=6)
        brotli_ratio = original_size / len(brotli_compressed)
        
        # Gzip compression
        gzip_compressed = gzip.compress(data, compresslevel=6)
        gzip_ratio = original_size / len(gzip_compressed)
        
        # Zlib compression
        zlib_compressed = zlib.compress(data, level=6)
        zlib_ratio = original_size / len(zlib_compressed)
        
        # Determine best compression
        results = {
            'Brotli': len(brotli_compressed),
            'Gzip': len(gzip_compressed), 
            'Zlib': len(zlib_compressed)
        }
        
        best_method = min(results.keys(), key=lambda x: results[x])
        
        print(f"{name:<12} | {original_size:<8} | {len(brotli_compressed):<5}({brotli_ratio:.1f}x) | {len(gzip_compressed):<5}({gzip_ratio:.1f}x) | {len(zlib_compressed):<5}({zlib_ratio:.1f}x) | {best_method}")
    
    # Speed benchmarks
    print("\n2. Speed Benchmarks:")
    
    # Create larger test data
    speed_test_data = ("Performance testing with various patterns. " * 500).encode('utf-8')
    iterations = 5
    
    print(f"Test data size: {len(speed_test_data):,} bytes")
    print(f"Iterations: {iterations}")
    print()
    
    # Brotli speed test
    brotli_times = []
    for _ in range(iterations):
        start = time.time()
        compressed = brotli.compress(speed_test_data, quality=4)
        brotli_times.append(time.time() - start)
    
    avg_brotli = sum(brotli_times) / iterations
    brotli_speed = len(speed_test_data) / avg_brotli / (1024*1024)  # MB/s
    
    print(f"Brotli (quality=4):")
    print(f"  Compression: {avg_brotli*1000:.2f}ms ({brotli_speed:.1f} MB/s)")
    print(f"  Compressed size: {len(compressed):,} bytes (ratio: {len(speed_test_data)/len(compressed):.2f}:1)")
    
    # Gzip speed test for comparison
    gzip_times = []
    for _ in range(iterations):
        start = time.time()
        gzip_compressed = gzip.compress(speed_test_data, compresslevel=4)
        gzip_times.append(time.time() - start)
    
    avg_gzip = sum(gzip_times) / iterations
    gzip_speed = len(speed_test_data) / avg_gzip / (1024*1024)
    
    print(f"\nGzip (level=4):")
    print(f"  Compression: {avg_gzip*1000:.2f}ms ({gzip_speed:.1f} MB/s)")
    print(f"  Compressed size: {len(gzip_compressed):,} bytes (ratio: {len(speed_test_data)/len(gzip_compressed):.2f}:1)")
    
    # Real-world application scenarios
    print("\n3. Real-World Application Scenarios:")
    
    # Web content compression
    print("Web Content:")
    html_content = """
    <!DOCTYPE html>
    <html><head><title>Sample Page</title>
    <style>body{font-family:Arial;margin:40px}.header{color:#333}</style>
    </head><body><div class="header"><h1>Welcome</h1></div>
    <div class="content"><p>Sample HTML content for compression testing.</p></div>
    </body></html>
    """ * 20
    
    html_bytes = html_content.encode('utf-8')
    html_compressed = brotli.compress(html_bytes, quality=6)
    html_ratio = len(html_bytes) / len(html_compressed)
    
    print(f"  HTML: {len(html_bytes):,} -> {len(html_compressed):,} bytes ({html_ratio:.1f}x reduction)")
    
    # API response compression
    api_response = {
        "status": "success",
        "data": [
            {"id": i, "username": f"user_{i}", "email": f"user_{i}@example.com", 
             "profile": {"age": 25 + i % 40, "city": ["NYC", "LA", "Chicago"][i % 3]}}
            for i in range(25)
        ],
        "meta": {"generated_at": "2024-01-01T00:00:00Z", "total": 25}
    }
    
    api_json = json.dumps(api_response, indent=2).encode('utf-8')
    api_compressed = brotli.compress(api_json, quality=6)
    api_ratio = len(api_json) / len(api_compressed)
    
    print(f"  JSON API: {len(api_json):,} -> {len(api_compressed):,} bytes ({api_ratio:.1f}x reduction)")
    
    # Log file compression
    log_entries = []
    for i in range(200):
        timestamp = f"2024-01-01 10:{i%60:02d}:{(i*13)%60:02d}"
        level = ["INFO", "WARN", "ERROR", "DEBUG"][i % 4]
        message = f"Event {i}: Processing request #{i} with status code {200 + i%100}."
        log_entries.append(f"[{timestamp}] {level}: {message}")
    
    log_content = '\n'.join(log_entries).encode('utf-8')
    log_compressed = brotli.compress(log_content, quality=6)
    log_ratio = len(log_content) / len(log_compressed)
    
    print(f"  Logs: {len(log_content):,} -> {len(log_compressed):,} bytes ({log_ratio:.1f}x reduction)")
    
    # Data integrity verification
    print("\n4. Data Integrity and Security:")
    
    critical_data = b"Important data that must maintain integrity through compression/decompression cycles." * 50
    
    # Create hash of original data
    original_hash = hashlib.sha256(critical_data).hexdigest()
    
    # Compress and decompress
    compressed_critical = brotli.compress(critical_data, quality=8)
    decompressed_critical = brotli.decompress(compressed_critical)
    
    # Verify hash after round-trip
    decompressed_hash = hashlib.sha256(decompressed_critical).hexdigest()
    
    print(f"  Original hash:     {original_hash[:32]}...")
    print(f"  Decompressed hash: {decompressed_hash[:32]}...")
    print(f"  Hash verification: {'✓ PASSED' if original_hash == decompressed_hash else '✗ FAILED'}")
    print(f"  Byte comparison: {'✓ PASSED' if critical_data == decompressed_critical else '✗ FAILED'}")
    
    # Memory efficiency analysis
    print("\n5. Memory Efficiency Analysis:")
    
    # Test with different data sizes
    sizes = [1024, 10*1024, 100*1024]  # 1KB to 100KB
    
    for size in sizes:
        test_data = b"M" * size  # Repetitive data for good compression
        
        start_time = time.time()
        compressed = brotli.compress(test_data, quality=6)
        compress_time = time.time() - start_time
        
        start_time = time.time()
        decompressed = brotli.decompress(compressed)
        decompress_time = time.time() - start_time
        
        ratio = len(test_data) / len(compressed)
        throughput = size / compress_time / (1024*1024) if compress_time > 0 else float('inf')
        
        size_label = f"{size//1024}KB"
        
        print(f"  {size_label:>4}: Ratio {ratio:.1f}:1 | "
              f"Compress {compress_time*1000:.2f}ms | "
              f"Throughput {throughput:.1f} MB/s")
    
    # Use case recommendations
    print("\n6. Use Case Recommendations:")
    
    use_cases = {
        "Web servers": "Quality 4-6 for good compression with reasonable speed",
        "File archival": "Quality 9-11 for maximum space savings",
        "Real-time apps": "Quality 1-3 for fastest compression",
        "API responses": "Quality 6 for balanced performance",
        "Log compression": "Quality 8-9 for high compression of repetitive data",
        "Asset delivery": "Quality 6-8 for static assets (CSS, JS, fonts)",
        "Database backups": "Quality 9-11 for maximum compression",
        "Streaming data": "Quality 4-6 with streaming classes if available"
    }
    
    print("Recommended quality levels by use case:")
    for use_case, recommendation in use_cases.items():
        print(f"  • {use_case:15}: {recommendation}")
    
    # Performance vs compression trade-off
    print("\n7. Performance vs Compression Trade-off:")
    
    trade_off_data = b"Trade-off analysis data for Brotli compression testing. " * 100
    
    print("Quality | Size    | Ratio | Compress | Decompress")
    print("--------|---------|-------|----------|----------")
    
    for quality in [1, 4, 6, 9, 11]:
        try:
            # Compression
            start = time.time()
            compressed = brotli.compress(trade_off_data, quality=quality)
            compress_ms = (time.time() - start) * 1000
            
            # Decompression
            start = time.time()
            decompressed = brotli.decompress(compressed)
            decompress_ms = (time.time() - start) * 1000
            
            ratio = len(trade_off_data) / len(compressed)
            integrity = trade_off_data == decompressed
            
            print(f"   {quality:2d}   | {len(compressed):7} | {ratio:5.1f} | {compress_ms:8.2f} | {decompress_ms:8.2f} | {'✓' if integrity else '✗'}")
            
        except Exception as e:
            print(f"   {quality:2d}   | Error: {e}")
            raise
    
    print("\n✓ Performance comparison and real-world applications completed successfully!")
    
except Exception as e:
    print(f"Error in performance analysis: {e}")
    raise AssertionError(
        f"brotli example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: brotli functionality test 5

In [ ]:
# Example 5: Comprehensive Brotli capabilities and ecosystem
import brotli
import inspect
import sys
import platform

print("=== Comprehensive Brotli Capabilities and Ecosystem ===")

try:
    # Module overview and capabilities
    print("1. Module Overview and Capabilities:")
    
    # Get all public attributes
    public_attrs = [attr for attr in dir(brotli) if not attr.startswith('_')]
    print(f"Public attributes ({len(public_attrs)}): {public_attrs}")
    
    # Version and build information
    version = getattr(brotli, '__version__', 'Unknown')
    print(f"\nBrotli version: {version}")
    
    # Module constants and modes
    print("\n2. Constants and Modes:")
    
    constants = {
        'MODE_GENERIC': getattr(brotli, 'MODE_GENERIC', None),
        'MODE_TEXT': getattr(brotli, 'MODE_TEXT', None), 
        'MODE_FONT': getattr(brotli, 'MODE_FONT', None),
        'DEFAULT_QUALITY': getattr(brotli, 'DEFAULT_QUALITY', None),
        'DEFAULT_WINDOW': getattr(brotli, 'DEFAULT_WINDOW', None),
        'MAX_QUALITY': getattr(brotli, 'MAX_QUALITY', None),
        'MIN_QUALITY': getattr(brotli, 'MIN_QUALITY', None)
    }
    
    for name, value in constants.items():
        if value is not None:
            print(f"  {name}: {value}")
        else:
            print(f"  {name}: Not available")
    
    # Function signatures and documentation
    print("\n3. Function Signatures:")
    
    functions = ['compress', 'decompress', 'Compressor', 'Decompressor']
    
    # brotli's compress() is a Python wrapper with an introspectable signature; decompress, Compressor
    # and Decompressor are C builtins with no __text_signature__ (inspect.signature raises ValueError),
    # and they document their signature in a "Signature:" section of the docstring instead.
    for func_name in functions:
        func = getattr(brotli, func_name)
        doc = inspect.getdoc(func)
        assert doc, f"brotli.{func_name} has no docstring"
        doc_lines = [line.strip() for line in doc.splitlines()]
        if "Signature:" in doc_lines:
            signature = doc_lines[doc_lines.index("Signature:") + 1]
        else:
            signature = f"{func_name}{inspect.signature(func)}"
        assert signature.startswith(func_name), f"unexpected signature for brotli.{func_name}: {signature!r}"
        print(f"  {signature}")
        print(f"    {doc[:80]}{'...' if len(doc) > 80 else ''}")
    
    # Compression mode testing if modes are available
    print("\n4. Compression Mode Testing:")
    
    test_data_types = {
        "Text": (b"This is plain text content for mode testing. " * 20, 'MODE_TEXT'),
        "Binary": (bytes(range(256)) * 2, 'MODE_GENERIC'),
        "Font-like": (b"\x00\x01\x02\x03" * 50, 'MODE_FONT')
    }
    
    for data_type, (data, suggested_mode) in test_data_types.items():
        print(f"\n  {data_type} data ({len(data)} bytes):")
        
        # Test with different modes if available
        modes_to_test = []
        if hasattr(brotli, 'MODE_GENERIC'):
            modes_to_test.append(('MODE_GENERIC', brotli.MODE_GENERIC))
        if hasattr(brotli, 'MODE_TEXT'):
            modes_to_test.append(('MODE_TEXT', brotli.MODE_TEXT))
        if hasattr(brotli, 'MODE_FONT'):
            modes_to_test.append(('MODE_FONT', brotli.MODE_FONT))
        
        if not modes_to_test:
            # Test without mode parameter
            compressed = brotli.compress(data, quality=6)
            ratio = len(data) / len(compressed)
            print(f"    Default mode: {len(compressed)} bytes (ratio: {ratio:.2f}:1)")
        else:
            for mode_name, mode_value in modes_to_test:
                try:
                    compressed = brotli.compress(data, quality=6, mode=mode_value)
                    ratio = len(data) / len(compressed)
                    recommended = "★" if suggested_mode == mode_name else " "
                    print(f"    {mode_name:12}{recommended}: {len(compressed)} bytes (ratio: {ratio:.2f}:1)")
                except Exception as e:
                    print(f"    {mode_name:12} : Error - {e}")
                    raise
    
    # Error handling and edge cases
    print("\n5. Error Handling and Edge Cases:")
    
    edge_cases = [
        (b"", "Empty data"),
        (b"x", "Single byte"),
        (b"x" * 100000, "Large repetitive (100KB)"),
        (bytes([255] * 1000), "High entropy data"),
        (b"\x00" * 1000, "All zeros")
    ]
    
    for data, description in edge_cases:
        try:
            # Test compression
            compressed = brotli.compress(data, quality=6)
            decompressed = brotli.decompress(compressed)
            
            integrity = data == decompressed
            ratio = len(data) / len(compressed) if len(compressed) > 0 else float('inf')
            
            print(f"  {description:22}: {len(data):6} -> {len(compressed):6} bytes | "
                  f"Ratio: {ratio:6.2f}:1 | {'✓' if integrity else '✗'}")
            
        except Exception as e:
            print(f"  {description:22}: Error - {e}")
            raise
    
    # Invalid data handling
    print("\n6. Invalid Data Handling:")
    
    invalid_cases = [
        (b"not compressed data", "Plain text as compressed"),
        (b"\x00\x01\x02\x03", "Random bytes"),
        (b"", "Empty compressed data"),
        (b"\xff" * 100, "High entropy noise")
    ]
    
    for invalid_data, description in invalid_cases:
        # The decoder must REJECT each input with brotli.error; accepting one is a failure.
        try:
            result = brotli.decompress(invalid_data)
        except brotli.error as e:
            assert "decoder failed" in str(e), f"{description}: unexpected brotli.error message: {e}"
            print(f"  {description:25}: {type(e).__name__} (expected)")
        else:
            raise AssertionError(f"{description}: brotli.decompress accepted invalid data ({len(result)} bytes)")
    
    # Ecosystem integration patterns
    print("\n7. Ecosystem Integration Patterns:")
    
    integration_examples = {
        "Web Frameworks": "Flask/Django response compression middleware",
        "File Processing": "Log file compression, data export optimization",
        "Network Protocols": "HTTP/2 compression, WebSocket message compression",
        "Data Storage": "Database backup compression, archive optimization",
        "Streaming Media": "Real-time data compression for streaming apps",
        "Mobile/IoT": "Bandwidth reduction for resource-constrained devices",
        "Gaming": "Asset compression for faster game content delivery",
        "CDN/Edge": "Content delivery optimization at edge locations"
    }
    
    print("Common use cases:")
    for category, description in integration_examples.items():
        print(f"  • {category:15}: {description}")
    
    # Performance characteristics summary
    print("\n8. Performance Characteristics Summary:")
    
    # Quick performance overview with different quality levels
    sample_data = b"Performance analysis sample data for Brotli testing. " * 50
    
    print(f"Sample data: {len(sample_data)} bytes")
    print("Quality level performance overview:")
    
    import time
    for quality in [1, 4, 6, 9, 11]:
        try:
            start_time = time.time()
            compressed = brotli.compress(sample_data, quality=quality)
            compress_time = time.time() - start_time
            
            start_time = time.time()
            decompressed = brotli.decompress(compressed)
            decompress_time = time.time() - start_time
            
            ratio = len(sample_data) / len(compressed)
            integrity = sample_data == decompressed
            
            print(f"  Quality {quality:2d}: {len(compressed):4} bytes | "
                  f"Ratio {ratio:4.1f}:1 | "
                  f"Compress {compress_time*1000:5.1f}ms | "
                  f"Decompress {decompress_time*1000:4.1f}ms | "
                  f"{'✓' if integrity else '✗'}")
            
        except Exception as e:
            print(f"  Quality {quality:2d}: Error - {e}")
            raise
    
    # Library and platform information
    print("\n9. Library and Platform Information:")
    
    print(f"Python version: {sys.version.split()[0]}")
    print(f"Platform: {platform.system()} {platform.machine()}")
    print(f"Brotli module location: {getattr(brotli, '__file__', 'Built-in/C extension')}")
    
    # Check implementation type
    if hasattr(brotli, '__file__'):
        if brotli.__file__.endswith('.so') or brotli.__file__.endswith('.pyd'):
            print("Implementation: Compiled C extension (optimized)")
        else:
            print("Implementation: Python module")
    else:
        print("Implementation: Built-in or C extension")
    
    # Feature availability summary
    print("\n10. Feature Availability Summary:")
    
    features = [
        ("Basic compress/decompress", hasattr(brotli, 'compress') and hasattr(brotli, 'decompress')),
        ("Streaming compression", hasattr(brotli, 'Compressor')),
        ("Streaming decompression", hasattr(brotli, 'Decompressor')),
        ("Quality parameter", True),  # Always available
        ("Window size (lgwin)", True),  # Test this separately
        ("Text mode", hasattr(brotli, 'MODE_TEXT')),
        ("Font mode", hasattr(brotli, 'MODE_FONT')),
        ("Generic mode", hasattr(brotli, 'MODE_GENERIC')),
        ("Constants", hasattr(brotli, 'MAX_QUALITY') or hasattr(brotli, 'DEFAULT_QUALITY'))
    ]
    
    # Test lgwin parameter availability
    lgwin_available = False
    try:
        brotli.compress(b"test", quality=6, lgwin=20)
        lgwin_available = True
    except TypeError:
        pass
    
    features[4] = ("Window size (lgwin)", lgwin_available)
    
    print("Feature availability:")
    for feature, available in features:
        status = "✓" if available else "✗"
        print(f"  {status} {feature}")
    
    # Usage recommendations
    print("\n11. Usage Recommendations:")
    
    recommendations = [
        "Quality 1-3: Use for real-time applications requiring fast compression",
        "Quality 4-6: Balanced compression ratio and speed for most web applications",
        "Quality 7-9: Higher compression for storage-optimized applications",
        "Quality 10-11: Maximum compression for archival and bandwidth-critical uses",
        "MODE_TEXT: Optimize for text-heavy content (HTML, JSON, logs)",
        "MODE_FONT: Optimize for font files and similar structured binary data",
        "Streaming: Use Compressor/Decompressor classes for large files",
        "Error handling: Always wrap compress/decompress in try-except blocks",
        "Web use: Excellent HTTP compression, often better than gzip",
        "Storage: Ideal for reducing backup sizes and archive storage costs"
    ]
    
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i:2d}. {rec}")
    
    # Algorithm comparison summary
    print("\n12. Algorithm Comparison Summary:")
    
    comparison_data = b"Algorithm comparison test data. " * 100
    
    algorithms = [
        ("Brotli", lambda d: brotli.compress(d, quality=6)),
        ("Gzip", lambda d: __import__('gzip').compress(d, compresslevel=6)),
        ("Zlib", lambda d: __import__('zlib').compress(d, level=6))
    ]
    
    print(f"Test data: {len(comparison_data)} bytes")
    print("Algorithm comparison:")
    
    for name, compress_func in algorithms:
        try:
            compressed = compress_func(comparison_data)
            ratio = len(comparison_data) / len(compressed)
            efficiency = (1 - len(compressed) / len(comparison_data)) * 100
            
            print(f"  {name:8}: {len(compressed):5} bytes | Ratio {ratio:.2f}:1 | Efficiency {efficiency:.1f}%")
            
        except Exception as e:
            print(f"  {name:8}: Error - {e}")
            raise
    
    print("\n✓ Comprehensive Brotli capabilities overview completed successfully!")
    
except Exception as e:
    print(f"Error in comprehensive overview: {e}")
    raise AssertionError(
        f"brotli example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e